In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('OneDrive/Desktop/FMCG_Project/outputs/retail_clean.csv')
df['invoicedate'] = pd.to_datetime(df['invoicedate'])

print(f"Rows: {len(df):,}")
print(f"Date range: {df['invoicedate'].min().date()} → {df['invoicedate'].max().date()}")

Rows: 805,549
Date range: 2009-12-01 → 2011-12-09


In [2]:
# In real companies this would be "today"
# We use the day after the last transaction in the dataset
reference_date = df['invoicedate'].max() + pd.Timedelta(days=1)
print(f"Reference date: {reference_date.date()}")

Reference date: 2011-12-10


In [3]:
rfm = df.groupby('customer_id').agg(
    recency   = ('invoicedate', lambda x: (reference_date - x.max()).days),
    frequency = ('invoice',     'nunique'),   # number of unique orders
    monetary  = ('revenue',     'sum')
).reset_index()

rfm.columns = ['customer_id', 'recency', 'frequency', 'monetary']
rfm['monetary'] = rfm['monetary'].round(2)

print(rfm.shape)
print(rfm.describe())

(5878, 4)
        customer_id      recency    frequency       monetary
count   5878.000000  5878.000000  5878.000000    5878.000000
mean   15315.313542   201.331916     6.289384    3018.616734
std     1715.572666   209.338707    13.009406   14737.731038
min    12346.000000     1.000000     1.000000       2.950000
25%    13833.250000    26.000000     1.000000     348.762500
50%    15314.500000    96.000000     3.000000     898.915000
75%    16797.750000   380.000000     7.000000    2307.090000
max    18287.000000   739.000000   398.000000  608821.650000


In [4]:
# Recency: LOWER days = BETTER, so score 5 = most recent
rfm['r_score'] = pd.qcut(rfm['recency'],   q=5, labels=[5,4,3,2,1])

# Frequency and Monetary: HIGHER = BETTER, so score 5 = highest
rfm['f_score'] = pd.qcut(rfm['frequency'].rank(method='first'), q=5, labels=[1,2,3,4,5])
rfm['m_score'] = pd.qcut(rfm['monetary'],  q=5, labels=[1,2,3,4,5])

# Convert to int for combining
rfm['r_score'] = rfm['r_score'].astype(int)
rfm['f_score'] = rfm['f_score'].astype(int)
rfm['m_score'] = rfm['m_score'].astype(int)

# Combined RFM score (string, used for segmentation)
rfm['rfm_score'] = rfm['r_score'].astype(str) + rfm['f_score'].astype(str) + rfm['m_score'].astype(str)

print(rfm.head(10))

   customer_id  recency  frequency  monetary  r_score  f_score  m_score  \
0        12346      326         12  77556.46        2        5        5   
1        12347        2          8   5633.32        5        4        5   
2        12348       75          5   2019.40        3        4        4   
3        12349       19          4   4428.69        5        3        5   
4        12350      310          1    334.40        2        1        2   
5        12351      375          1    300.93        2        1        2   
6        12352       36         10   2849.84        4        5        4   
7        12353      204          2    406.76        2        2        2   
8        12354      232          1   1079.40        2        1        3   
9        12355      214          2    947.61        2        2        3   

  rfm_score  
0       255  
1       545  
2       344  
3       535  
4       212  
5       212  
6       454  
7       222  
8       213  
9       223  


In [5]:
def assign_segment(row):
    r, f = row['r_score'], row['f_score']
    
    if r >= 4 and f >= 4:
        return 'Champion'
    elif r >= 3 and f >= 3:
        return 'Loyal'
    elif r >= 4 and f <= 2:
        return 'New Customer'
    elif r >= 3 and f <= 2:
        return 'Potential Loyalist'
    elif r == 2 and f >= 3:
        return 'At Risk'
    elif r <= 2 and f <= 2:
        return 'Lost'
    else:
        return 'Needs Attention'

rfm['segment'] = rfm.apply(assign_segment, axis=1)

print(rfm['segment'].value_counts())

segment
Lost                  1523
Champion              1482
Loyal                 1221
At Risk                551
New Customer           443
Potential Loyalist     385
Needs Attention        273
Name: count, dtype: int64


In [6]:
segment_summary = rfm.groupby('segment').agg(
    customer_count = ('customer_id', 'count'),
    avg_recency    = ('recency',     'mean'),
    avg_frequency  = ('frequency',   'mean'),
    avg_monetary   = ('monetary',    'mean'),
    total_revenue  = ('monetary',    'sum')
).round(2).reset_index()

segment_summary['revenue_pct'] = (
    segment_summary['total_revenue'] / segment_summary['total_revenue'].sum() * 100
).round(1)

segment_summary = segment_summary.sort_values('total_revenue', ascending=False)
print(segment_summary.to_string(index=False))

           segment  customer_count  avg_recency  avg_frequency  avg_monetary  total_revenue  revenue_pct
          Champion            1482        20.37          15.66       8294.97    12293138.65         69.3
             Loyal            1221        78.57           5.42       2087.58     2548937.65         14.4
           At Risk             551       303.15           5.19       2126.25     1171564.34          6.6
              Lost            1523       459.28           1.25        438.03      667121.92          3.8
   Needs Attention             273       502.21           4.43       1693.81      462411.15          2.6
      New Customer             443        28.11           1.46        890.83      394638.61          2.2
Potential Loyalist             385       107.11           1.36        534.07      205616.84          1.2


In [7]:
# Full RFM table (for Power BI)
rfm.to_csv('OneDrive/Desktop/FMCG_Project/outputs/rfm_scores.csv', index=False)

# Segment summary (for Power BI KPI cards)
segment_summary.to_csv('OneDrive/Desktop/FMCG_Project/outputs/rfm_segment_summary.csv', index=False)

print("✓ Saved rfm_scores.csv")
print("✓ Saved rfm_segment_summary.csv")
print(f"\nTotal customers scored: {len(rfm):,}")

✓ Saved rfm_scores.csv
✓ Saved rfm_segment_summary.csv

Total customers scored: 5,878


In [8]:
# Are our top champions businesses or individuals?
print(rfm[rfm['segment']=='Champion'].sort_values('frequency', ascending=False).head(5)[
    ['customer_id','recency','frequency','monetary']
])

      customer_id  recency  frequency   monetary
2538        14911        1        398  295972.63
400         12748        1        336   56599.39
5433        17841        2        211   70884.07
2935        15311        1        208  116771.16
739         13089        3        203  116737.86
